# Mistral-7B Fine-tuning with LoRA & QLoRA
Healthcare chatbot fine-tuning on HealthCareMagic dataset

In [1]:
!nvidia-smi

Wed Jul 15 04:14:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:1E.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl gradio huggingface_hub

In [3]:
import transformers
import datasets
import peft
import bitsandbytes

In [4]:
import os
from getpass import getpass
from huggingface_hub import login

# Works on Colab, Lightning AI Studio, or any Jupyter environment.
# Set HF_TOKEN as an environment variable beforehand if you don't want to be prompted:
#   export HF_TOKEN=hf_xxx   (in a terminal, before launching the notebook)
hf_token = os.environ.get("HF_TOKEN") or getpass("Enter your Hugging Face token: ")
login(token=hf_token)


Enter your Hugging Face token:  ········


In [5]:
from datasets import load_dataset

# Load HealthCareMagic healthcare dataset
dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")

print(dataset)
print("\nFirst example:")
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 112165
    })
})

First example:
{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!', 'output': 'Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of perip

In [6]:
def format_instruction(example):
    """
    Format dataset examples into instruction style
    for Mistral fine-tuning.
    """
    return {
        "text": f"""<s>[INST] You are a helpful medical assistant.
{example['instruction']}

Patient: {example['input']} [/INST]

Doctor: {example['output']} </s>"""
    }

# Apply formatting to entire dataset
formatted_dataset = dataset.map(format_instruction)

print("Formatting done! ✅")
print("\nFirst formatted example:")
print(formatted_dataset['train'][0]['text'])

Formatting done! ✅

First formatted example:
<s>[INST] You are a helpful medical assistant.
If you are a doctor, please answer the medical questions based on the patient's description.

Patient: I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc! [/INST]

Doctor: Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is di

In [7]:
def is_valid_example(example):
    """
    Filter out low quality examples from dataset.
    """
    # Remove empty inputs or outputs
    if not example['input'] or not example['output']:
        return False

    # Remove very short responses (low quality)
    if len(example['output'].split()) < 20:
        return False

    # Remove very long responses (kept short to fit training time budget)
    if len(example['text'].split()) > 256:
        return False

    return True

# Apply filtering
cleaned_dataset = formatted_dataset.filter(is_valid_example)

print(f"Original dataset size: {len(formatted_dataset['train'])}")
print(f"Cleaned dataset size: {len(cleaned_dataset['train'])}")
print(f"Removed: {len(formatted_dataset['train']) - len(cleaned_dataset['train'])} examples")


Filter:   0%|          | 0/112165 [00:00<?, ? examples/s]

Original dataset size: 112165
Cleaned dataset size: 91961
Removed: 20204 examples


In [8]:
# Split dataset into train and test
split_dataset = cleaned_dataset['train'].train_test_split(
    test_size=0.2,  # 20% for testing
    seed=42         # for reproducibility
)

# Cap dataset size to fit training within a ~5 hour budget on a free-tier T4.
# NOTE: train_test_split() already shuffles internally, so we just .select()
# the first N rows here instead of shuffling again -- a second .shuffle() on
# a freshly-filtered dataset can be very slow (full Arrow table reindex).
# Comment these two lines out later if you want a full run with more time/GPU.
MAX_TRAIN_EXAMPLES = 15000
MAX_EVAL_EXAMPLES = 1500

split_dataset['train'] = split_dataset['train'].select(
    range(min(MAX_TRAIN_EXAMPLES, len(split_dataset['train'])))
)
split_dataset['test'] = split_dataset['test'].select(
    range(min(MAX_EVAL_EXAMPLES, len(split_dataset['test'])))
)

print(split_dataset)
print(f"\nTraining examples: {len(split_dataset['train'])}")
print(f"Testing examples: {len(split_dataset['test'])}")


DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 15000
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 1500
    })
})

Training examples: 15000
Testing examples: 1500


In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Model name
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Required when using gradient checkpointing during training,
# otherwise you'll get incorrect gradients / a cache warning.
model.config.use_cache = False

print("Model loaded successfully!")
print(f"Model device: {next(model.parameters()).device}")


Loading tokenizer...
Loading model in 4-bit...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded successfully!
Model device: cuda:0


## Tokenize Dataset

In [10]:
# Tokenize the dataset
def tokenize_function(examples):
    """
    Tokenize the text examples.
    """
    tokenized = tokenizer(
        examples['text'],
        padding='max_length',
        max_length=256,     # shortened from 512 to speed up training
        truncation=True,
        return_tensors=None
    )

    # Set labels to input_ids for language modeling
    tokenized['labels'] = tokenized['input_ids'].copy()

    return tokenized

print("Tokenizing training dataset...")
tokenized_train = split_dataset['train'].map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=['text']
)

print("Tokenizing test dataset...")
tokenized_test = split_dataset['test'].map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=['text']
)

print("✅ Tokenization complete!")
print(f"Train dataset: {len(tokenized_train)}")
print(f"Test dataset: {len(tokenized_test)}")
print(f"\nExample tokens: {tokenized_train[0]}")


Tokenizing training dataset...


Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Tokenizing test dataset...


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

✅ Tokenization complete!
Train dataset: 15000
Test dataset: 1500

Example tokens: {'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'Hi ya  i have a 12 year old boy  since last week he has got tierd very quickly and slept all day monday, he was sick all so sunday night and monday when he was a wake. I sent him back to school today because he was a lot brighter this morning but he has gone back down again since coming home . He had scarlett fever about a year and a half ago  could this be this coming back ?', 'output': 'Hi The symptoms described here are not very specific to scarlet fever. But any bacterial or viral infection may start with almost similar symptoms initially. You can start with a tab paracetamol(500\\u00a0mg) for the symptoms mentioned above and can repeat after 4-6 hours if symptoms appear again .let him take rest if he feels to do so. Also give him lot of oral fluids/liquids (like juices, soups, mil

## QLoRA Fine-tuning Configuration

In [11]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# LoRA config for QLoRA fine-tuning
lora_config = LoraConfig(
    r=8,                           # LoRA rank
    lora_alpha=16,                 # LoRA alpha (2x rank is common)
    target_modules=["q_proj", "v_proj"],  # Target attention modules
    lora_dropout=0.05,             # Dropout for LoRA layers
    bias="none",                   # No bias adaptation
    task_type="CAUSAL_LM"
)

print("LoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Dropout: {lora_config.lora_dropout}")
print(f"  Target modules: {lora_config.target_modules}")

# Get PEFT model
model = get_peft_model(model, lora_config)

# Print trainable parameters
def print_trainable_parameters(model):
    """
    Print number of trainable parameters.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"trainable params: {trainable_params:,d} || all params: {all_param:,d} || trainable%: {100 * trainable_params / all_param:.2f}")

print("\n📊 Model Parameters:")
print_trainable_parameters(model)


LoRA Configuration:
  Rank (r): 8
  Alpha: 16
  Dropout: 0.05
  Target modules: {'v_proj', 'q_proj'}

📊 Model Parameters:
trainable params: 3,407,872 || all params: 3,755,479,040 || trainable%: 0.09


## Training Configuration

In [13]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import os

output_dir = "./mistral-7b-healthcare-qlora"
os.makedirs(output_dir, exist_ok=True)

# With 15,000 train examples, batch_size=8, grad_accum=2 (effective batch=16):
#   15000 / 16 ≈ 938 steps for 1 epoch
# max_steps below is a hard safety cap in case per-step speed is worse than expected.
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    max_steps=900,                 # hard stop -- guarantees the run fits your time budget
    warmup_steps=50,
    logging_steps=25,
    save_strategy="steps",
    save_steps=150,                # less frequent than before -> less checkpoint I/O overhead
    save_total_limit=3,            # keep disk usage bounded; only last 3 checkpoints
    eval_strategy="steps",
    eval_steps=150,                # less frequent eval -> less overhead
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=2e-4,
    weight_decay=0.01,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    fp16=True,
    seed=42,
)


## Initialize Trainer

In [14]:
# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal language modeling (not masked)
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    callbacks=[
        # Early stopping callback (optional)
        # EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)
    ]
)

print("✅ Trainer initialized successfully!")

✅ Trainer initialized successfully!


## Start Fine-tuning

In [15]:
import glob

# If training was interrupted (spot/interruptible GPU on Lightning AI, Colab
# disconnect, etc.), resume from the latest saved checkpoint instead of
# starting over from scratch.
existing_checkpoints = sorted(
    glob.glob(f"{output_dir}/checkpoint-*"),
    key=lambda p: int(p.split("-")[-1])
)
resume_from = existing_checkpoints[-1] if existing_checkpoints else None

if resume_from:
    print(f"🔄 Resuming training from checkpoint: {resume_from}")
else:
    print("🔥 Starting QLoRA fine-tuning from scratch...")

train_result = trainer.train(resume_from_checkpoint=resume_from)

print("\n✅ Training completed!")
print(f"Training loss: {train_result.training_loss:.4f}")


🔥 Starting QLoRA fine-tuning from scratch...


Step,Training Loss,Validation Loss
150,2.004402,2.011425
300,1.900276,1.907255
450,1.880250,1.885339
600,1.902963,1.870443
750,1.833618,1.863414
900,1.833041,1.861842



✅ Training completed!
Training loss: 1.9663


## Evaluate Model

In [16]:
import math

# Evaluate on test set
print("📊 Evaluating on test set...")
eval_result = trainer.evaluate()

perplexity = math.exp(eval_result["eval_loss"])

print("\nEvaluation Results:")
print(f"  Eval Loss: {eval_result['eval_loss']:.4f}")
print(f"  Perplexity: {perplexity:.2f}")


📊 Evaluating on test set...


Training Loss,Validation Loss,Step
1.833041,1.861842,900



Evaluation Results:
  Eval Loss: 1.8618
  Perplexity: 6.44


## Save Model

In [17]:
# Save the LoRA adapter
print("💾 Saving LoRA adapter...")
model.save_pretrained(f"{output_dir}/final-adapter")

# Save tokenizer
tokenizer.save_pretrained(f"{output_dir}/final-adapter")

print(f"✅ Model saved to {output_dir}/final-adapter")

💾 Saving LoRA adapter...
✅ Model saved to ./mistral-7b-healthcare-qlora/final-adapter


In [21]:
import gc
import torch

# Free GPU memory from the training run before loading a fresh copy for merging
try:
    del trainer
except NameError:
    pass
try:
    del model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

GPU memory allocated: 0.54 GB
GPU memory reserved: 4.66 GB


In [22]:
import os
from peft import AutoPeftModelForCausalLM

offload_dir = "./offload_tmp"
os.makedirs(offload_dir, exist_ok=True)

# Load the trained LoRA model
model_path = f"{output_dir}/final-adapter"
qlora_model = AutoPeftModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.float16,
    offload_folder=offload_dir,   # allows disk offload if GPU memory is still tight
)

# Merge LoRA weights with base model (if you want a single model file)
print("Merging LoRA weights with base model...")
merged_model = qlora_model.merge_and_unload()

# Save merged model
merged_model.save_pretrained(f"{output_dir}/merged-model")
tokenizer.save_pretrained(f"{output_dir}/merged-model")
print(f"✅ Merged model saved to {output_dir}/merged-model")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Merging LoRA weights with base model...


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/modeling_utils.py:3518: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (50GB default)
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Merged model saved to ./mistral-7b-healthcare-qlora/merged-model


## Load and Merge Model (Optional)

## Test Inference

In [24]:
# Test inference with the fine-tuned model
from transformers import pipeline

# NOTE: merged_model was already placed on GPU via device_map="auto" when loaded,
# so we must NOT also pass device=0 to pipeline() -- that raises a conflict error.
generator = pipeline(
    'text-generation',
    model=merged_model,
    tokenizer=tokenizer,
)

# Test prompt
test_prompt = """<s>[INST] You are a helpful medical assistant.
I have been experiencing persistent headaches for the past week. What could be the cause?

Patient: I wake up with a throbbing headache every morning. [/INST]

Doctor:"""

print("Testing inference...\n")
response = generator(
    test_prompt,
    max_new_tokens=200,
    do_sample=True,              # <-- required for temperature/top_p to actually take effect
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.3,      # <-- discourages repeated phrases
    no_repeat_ngram_size=3,      # <-- hard-blocks exact 3-word repeats (fixes "Thanks. Thanks. Thanks.")
    num_return_sequences=1,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

print("Generated Response:")
print(response[0]['generated_text'][len(test_prompt):])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty', 'temperature', 'do_sample', 'top_p', 'eos_token_id', 'no_repeat_ngram_size', 'num_return_sequences', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Testing inference...

Generated Response:
Hi, Thanks for writing to us, Headaches can occur due to many reasons like stress or anxiety, migraine etc. Some of them may require medications and some don't need any treatment at all. If your symptoms persist then you should consult an ENT specialist who will examine your nose to exclude nasal sinusitis as one commonest reason behind it is blocked nose because of which there is increased pressure in sinuses leading to severe headache.  Hope this helps, take care Chat Doctor. .  Consulted Physician.  Regards Jay Instructor.  Wish you good health.  Thank you.  Welcome again next time.  Migration from another site.  Have a great day!   Good luck.  Best wishes, Jay Toxicologist.  (Mentor)  From India.  Recommended physician.  Treatment 1-Nasal saline drops application2 -Steam inhalation3-Table


## Upload to Hugging Face Hub (Optional)

In [25]:
# Upload the adapter to Hugging Face Hub
repo_name = "d1wash/mistral-7b-healthcare-qlora"

print(f"Uploading to Hub as {repo_name}...")
qlora_model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f"✅ Model uploaded to https://huggingface.co/{repo_name}")

Uploading to Hub as d1wash/mistral-7b-healthcare-qlora...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

✅ Model uploaded to https://huggingface.co/d1wash/mistral-7b-healthcare-qlora
